# 72. The Capacitated VRP (CVRP)

## Tier 1: The Pen & Paper Method (Mathematical Formulation)

### Key assumptions
- Single depot with multiple vehicles
- Each vehicle has fixed capacity constraint
- Each customer must be served exactly once
- Vehicles must return to depot
- Objective is to minimize total travel distance

### Approach (step-by-step)
1. **Problem Definition**: Define sets, parameters, and decision variables
2. **Mathematical Model**: Formulate objective function and constraints
3. **Solution Method**: Use mixed-integer programming with pulp solver
4. **Analysis**: Extract and interpret optimal solution
5. **Visualization**: Display routes and performance metrics

### What to look for in the results
- Optimal vehicle-customer assignments
- Route sequences for each vehicle
- Total distance traveled
- Capacity utilization for each vehicle
- Computational performance metrics

### Concrete example (from the source)
Small instance with 5 customers and 3 vehicles, each with capacity Q = 6.
Customer demands are q = [2, 3, 1, 4, 2].
The optimal solution assigns customers 1 and 2 to vehicle 1 (total demand = 5),
customers 3 and 5 to vehicle 2 (total demand = 3), and customer 4 to vehicle 3 (total demand = 4).

In [1]:
# Import required libraries for mathematical optimization
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pulp import *
import itertools
import warnings
warnings.filterwarnings('ignore')

# Set random seed for reproducibility
np.random.seed(42)

print("Libraries imported successfully for CVRP mathematical formulation")

In [2]:
# Define CVRP problem data structure
class CVRPInstance:
    """Class to represent a CVRP instance with all necessary data"""
    
    def __init__(self, num_customers, num_vehicles, vehicle_capacity, 
                 customer_demands, customer_locations, depot_location=None):
        self.num_customers = num_customers
        self.num_vehicles = num_vehicles
        self.vehicle_capacity = vehicle_capacity
        self.customer_demands = customer_demands
        self.customer_locations = customer_locations
        
        # Set depot at origin if not specified
        if depot_location is None:
            self.depot_location = (0, 0)
        else:
            self.depot_location = depot_location
        
        # Calculate distance matrix
        self.distance_matrix = self._calculate_distance_matrix()
    
    def _calculate_distance_matrix(self):
        """Calculate Euclidean distance matrix between all points"""
        # Combine depot and customer locations
        all_locations = [self.depot_location] + self.customer_locations
        n_points = len(all_locations)
        
        # Initialize distance matrix
        dist_matrix = np.zeros((n_points, n_points))
        
        # Calculate pairwise distances
        for i in range(n_points):
            for j in range(n_points):
                if i != j:
                    dist_matrix[i][j] = np.sqrt(
                        (all_locations[i][0] - all_locations[j][0])**2 + 
                        (all_locations[i][1] - all_locations[j][1])**2
                    )
        
        return dist_matrix

print("CVRP instance class defined successfully")

In [ ]:
# Create the concrete example from the source material
# 5 customers, 3 vehicles, capacity = 6
customer_demands = [2, 3, 1, 4, 2]  # q = [2, 3, 1, 4, 2]
customer_locations = [(5, 10), (15, 5), (10, 15), (20, 10), (8, 12)]

# Create CVRP instance
cvrp_instance = CVRPInstance(
    num_customers=5,
    num_vehicles=3,
    vehicle_capacity=6,
    customer_demands=customer_demands,
    customer_locations=customer_locations
)

print(f"CVRP Instance Created:")
print(f"- Customers: {cvrp_instance.num_customers}")
print(f"- Vehicles: {cvrp_instance.num_vehicles}")
print(f"- Vehicle Capacity: {cvrp_instance.vehicle_capacity}")
print(f"- Customer Demands: {cvrp_instance.customer_demands}")
print(f"- Total Demand: {sum(cvrp_instance.customer_demands)}")
print(f"- Total Capacity: {cvrp_instance.num_vehicles * cvrp_instance.vehicle_capacity}")

In [3]:
# Mathematical formulation using Mixed-Integer Programming
def solve_cvrp_mip(cvrp_instance, time_limit=60):
    """Solve CVRP using Mixed-Integer Programming formulation"""
    
    # Create optimization problem
    prob = LpProblem("CVRP_MIP", LpMinimize)
    
    # Define sets
    customers = range(1, cvrp_instance.num_customers + 1)  # 1 to n
    vehicles = range(1, cvrp_instance.num_vehicles + 1)     # 1 to m
    all_nodes = [0] + list(customers)  # 0 (depot) + customers
    
    # Decision variables
    # x[i][j][k] = 1 if vehicle k travels from i to j
    x = {}
    for k in vehicles:
        for i in all_nodes:
            for j in all_nodes:
                if i != j:
                    x[(i, j, k)] = LpVariable(f"x_{i}_{j}_{k}", cat='Binary')
    
    # y[i][k] = 1 if customer i is served by vehicle k
    y = {}
    for i in customers:
        for k in vehicles:
            y[(i, k)] = LpVariable(f"y_{i}_{k}", cat='Binary')
    
    # Objective function: minimize total distance
    prob += lpSum(
        cvrp_instance.distance_matrix[i][j] * x[(i, j, k)]
        for k in vehicles
        for i in all_nodes
        for j in all_nodes
        if i != j
    ), "Total_Distance"
    
    # Constraints
    
    # 1. Each customer must be served exactly once
    for i in customers:
        prob += lpSum(y[(i, k)] for k in vehicles) == 1, f"Serve_Customer_{i}"
    
    # 2. Each vehicle must leave the depot exactly once
    for k in vehicles:
        prob += lpSum(x[(0, j, k)] for j in customers) == 1, f"Leave_Depot_{k}"
    
    # 3. Each vehicle must return to the depot exactly once
    for k in vehicles:
        prob += lpSum(x[(i, 0, k)] for i in customers) == 1, f"Return_Depot_{k}"
    
    # 4. Flow conservation: if vehicle serves customer, it must enter and exit
    for i in customers:
        for k in vehicles:
            prob += lpSum(x[(j, i, k)] for j in all_nodes if j != i) == y[(i, k)], f"Enter_Customer_{i}_Vehicle_{k}"
            prob += lpSum(x[(i, j, k)] for j in all_nodes if j != i) == y[(i, k)], f"Exit_Customer_{i}_Vehicle_{k}"
    
    # 5. Capacity constraint for each vehicle
    for k in vehicles:
        prob += lpSum(
            cvrp_instance.customer_demands[i-1] * y[(i, k)] for i in customers
        ) <= cvrp_instance.vehicle_capacity, f"Capacity_Vehicle_{k}"
    
    # 6. Subtour elimination (MTZ formulation)
    # Add auxiliary variables for MTZ
    u = {}
    for i in customers:
        u[i] = LpVariable(f"u_{i}", lowBound=0, cat='Continuous')
    
    for i in customers:
        for j in customers:
            if i != j:
                for k in vehicles:
                    prob += u[i] - u[j] + cvrp_instance.num_customers * x[(i, j, k)] <= cvrp_instance.num_customers - 1, f"Subtour_Elimination_{i}_{j}_{k}"
    
    # Set solver parameters
    solver = PULP_CBC_CMD(timeLimit=time_limit, msg=False)
    
    # Solve the problem
    print("Solving CVRP MIP...")
    result = prob.solve(solver)
    
    return prob, result, x, y

print("MIP formulation function defined successfully")

In [ ]:
# Solve the CVRP instance
prob, result, x_vars, y_vars = solve_cvrp_mip(cvrp_instance)

# Check solution status
print(f"\nSolution Status: {LpStatus[result]}")
print(f"Objective Value (Total Distance): {prob.objective.value():.2f}")

In [ ]:
# Extract and interpret the solution
def extract_solution(cvrp_instance, x_vars, y_vars):
    """Extract routes and assignments from the solution"""
    
    customers = range(1, cvrp_instance.num_customers + 1)
    vehicles = range(1, cvrp_instance.num_vehicles + 1)
    all_nodes = [0] + list(customers)
    
    # Extract customer assignments
    assignments = {}
    for k in vehicles:
        assignments[k] = []
        for i in customers:
            if y_vars[(i, k)].value() == 1:
                assignments[k].append(i)
    
    # Extract routes for each vehicle
    routes = {}
    for k in vehicles:
        if assignments[k]:  # If vehicle has customers assigned
            route = [0]  # Start from depot
            current = 0
            
            # Build route step by step
            while len(route) < len(assignments[k]) + 1:
                found = False
                for j in assignments[k]:
                    if j not in route[1:]:  # Not already visited
                        if x_vars[(current, j, k)].value() == 1:
                            route.append(j)
                            current = j
                            found = True
                            break
                
                # If no direct edge found, try to find any valid next node
                if not found:
                    for j in all_nodes:
                        if j != current and x_vars[(current, j, k)].value() == 1:
                            if j in assignments[k] and j not in route[1:]:
                                route.append(j)
                                current = j
                                found = True
                                break
            
            route.append(0)  # Return to depot
            routes[k] = route
        else:
            routes[k] = [0, 0]  # Empty route: depot to depot
    
    return assignments, routes

# Extract solution
assignments, routes = extract_solution(cvrp_instance, x_vars, y_vars)

print("Solution Extracted:")
for k in range(1, cvrp_instance.num_vehicles + 1):
    demand_served = sum(cvrp_instance.customer_demands[i-1] for i in assignments[k])
    print(f"\nVehicle {k}:")
    print(f"  - Customers served: {assignments[k]}")
    print(f"  - Route: {routes[k]}")
    print(f"  - Demand served: {demand_served}/{cvrp_instance.vehicle_capacity}")
    print(f"  - Capacity utilization: {demand_served/cvrp_instance.vehicle_capacity*100:.1f}%")

In [ ]:
# Calculate route distances and performance metrics
def calculate_route_metrics(cvrp_instance, routes):
    """Calculate distance and performance metrics for each route"""
    
    metrics = {}
    total_distance = 0
    
    for k, route in routes.items():
        route_distance = 0
        
        for i in range(len(route) - 1):
            from_node = route[i]
            to_node = route[i + 1]
            route_distance += cvrp_instance.distance_matrix[from_node][to_node]
        
        metrics[k] = {
            'route': route,
            'distance': route_distance,
            'customers_served': len([c for c in route if c != 0]),
            'demand_served': sum(cvrp_instance.customer_demands[c-1] for c in route if c != 0)
        }
        
        total_distance += route_distance
    
    return metrics, total_distance

# Calculate metrics
route_metrics, total_distance = calculate_route_metrics(cvrp_instance, routes)

print("\nRoute Performance Metrics:")
print(f"{'Vehicle':<8} {'Route':<20} {'Distance':<10} {'Customers':<10} {'Demand':<8} {'Util%':<8}")
print("-" * 70)

for k, metrics in route_metrics.items():
    util_pct = metrics['demand_served'] / cvrp_instance.vehicle_capacity * 100
    route_str = '->'.join(map(str, metrics['route']))
    print(f"{k:<8} {route_str:<20} {metrics['distance']:<10.2f} {metrics['customers_served']:<10} {metrics['demand_served']:<8} {util_pct:<8.1f}")

print(f"\nTotal Distance: {total_distance:.2f}")
print(f"Total Customers Served: {sum(m['customers_served'] for m in route_metrics.values())}")
print(f"Total Demand Served: {sum(m['demand_served'] for m in route_metrics.values())}")

In [ ]:
# Visualization of CVRP solution
def visualize_cvrp_solution(cvrp_instance, routes, assignments):
    """Create comprehensive visualization of the CVRP solution"""
    
    fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(15, 12))
    
    # Color palette for vehicles
    colors = ['red', 'blue', 'green', 'orange', 'purple', 'brown', 'pink', 'gray', 'olive', 'cyan']
    
    # Plot 1: Route visualization
    ax1.set_title('CVRP Optimal Routes', fontsize=14, fontweight='bold')
    
    # Plot depot
    ax1.plot(cvrp_instance.depot_location[0], cvrp_instance.depot_location[1], 
            'ks', markersize=15, label='Depot')
    
    # Plot customers with demand labels
    for i, (x, y) in enumerate(cvrp_instance.customer_locations):
        customer_id = i + 1
        demand = cvrp_instance.customer_demands[i]
        ax1.plot(x, y, 'o', markersize=8, color='black')
        ax1.annotate(f'C{customer_id}\n(d={demand})', (x, y), 
                    xytext=(5, 5), textcoords='offset points', fontsize=8)
    
    # Plot routes
    for k, route in routes.items():
        if len(route) > 2:  # Non-empty route
            color = colors[(k-1) % len(colors)]
            route_coords = []
            
            for node in route:
                if node == 0:
                    route_coords.append(cvrp_instance.depot_location)
                else:
                    route_coords.append(cvrp_instance.customer_locations[node-1])
            
            # Plot route path
            for i in range(len(route_coords) - 1):
                x_coords = [route_coords[i][0], route_coords[i+1][0]]
                y_coords = [route_coords[i][1], route_coords[i+1][1]]
                ax1.plot(x_coords, y_coords, color=color, linewidth=2, 
                        alpha=0.7, label=f'Vehicle {k}')
    
    ax1.set_xlabel('X Coordinate')
    ax1.set_ylabel('Y Coordinate')
    ax1.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    ax1.grid(True, alpha=0.3)
    
    # Plot 2: Vehicle capacity utilization
    ax2.set_title('Vehicle Capacity Utilization', fontsize=14, fontweight='bold')
    
    vehicles = list(range(1, cvrp_instance.num_vehicles + 1))
    capacities = []
    demands_served = []
    
    for k in vehicles:
        demand = sum(cvrp_instance.customer_demands[i-1] for i in assignments[k])
        capacities.append(cvrp_instance.vehicle_capacity)
        demands_served.append(demand)
    
    x_pos = np.arange(len(vehicles))
    width = 0.35
    
    ax2.bar(x_pos - width/2, capacities, width, label='Capacity', alpha=0.7, color='lightblue')
    ax2.bar(x_pos + width/2, demands_served, width, label='Demand Served', alpha=0.7, color='darkblue')
    
    ax2.set_xlabel('Vehicle')
    ax2.set_ylabel('Capacity Units')
    ax2.set_xticks(x_pos)
    ax2.set_xticklabels(vehicles)
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    
    # Plot 3: Distance contribution by vehicle
    ax3.set_title('Distance Contribution by Vehicle', fontsize=14, fontweight='bold')
    
    vehicle_distances = []
    for k, metrics in route_metrics.items():
        vehicle_distances.append(metrics['distance'])
    
    bars = ax3.bar(vehicles, vehicle_distances, color=colors[:len(vehicles)], alpha=0.7)
    ax3.set_xlabel('Vehicle')
    ax3.set_ylabel('Distance Traveled')
    ax3.grid(True, alpha=0.3)
    
    # Add value labels on bars
    for bar, dist in zip(bars, vehicle_distances):
        height = bar.get_height()
        ax3.text(bar.get_x() + bar.get_width()/2., height + 0.5,
                f'{dist:.1f}', ha='center', va='bottom', fontsize=10)
    
    # Plot 4: Solution summary statistics
    ax4.set_title('Solution Summary', fontsize=14, fontweight='bold')
    ax4.axis('off')
    
    # Create summary text
    summary_text = f"""
    CVRP Solution Statistics:
    ─────────────────────
    • Total Distance: {total_distance:.2f} units
    • Customers Served: {sum(m['customers_served'] for m in route_metrics.values())}
    • Total Demand: {sum(m['demand_served'] for m in route_metrics.values())}
    • Fleet Capacity: {cvrp_instance.num_vehicles * cvrp_instance.vehicle_capacity}
    • Capacity Utilization: {sum(m['demand_served'] for m in route_metrics.values())/(cvrp_instance.num_vehicles * cvrp_instance.vehicle_capacity)*100:.1f}%
    • Average Distance/Vehicle: {total_distance/cvrp_instance.num_vehicles:.2f}
    • Solution Status: Optimal
    """
    
    ax4.text(0.1, 0.5, summary_text, fontsize=11, verticalalignment='center',
             fontfamily='monospace', bbox=dict(boxstyle='round', facecolor='lightgray', alpha=0.8))
    
    plt.tight_layout()
    plt.show()

# Visualize the solution
visualize_cvrp_solution(cvrp_instance, routes, assignments)

In [ ]:
# Sensitivity analysis with different capacity values
def sensitivity_analysis_capacity(cvrp_instance):
    """Analyze how solution changes with different vehicle capacities"""
    
    original_capacity = cvrp_instance.vehicle_capacity
    capacity_range = [4, 5, 6, 8, 10]
    
    results = []
    
    print("\nSensitivity Analysis: Impact of Vehicle Capacity")
    print("=" * 50)
    print(f"{'Capacity':<10} {'Distance':<10} {'Vehicles Used':<15} {'Avg Util%':<10}")
    print("-" * 50)
    
    for capacity in capacity_range:
        # Create new instance with different capacity
        temp_instance = CVRPInstance(
            num_customers=cvrp_instance.num_customers,
            num_vehicles=cvrp_instance.num_vehicles,
            vehicle_capacity=capacity,
            customer_demands=cvrp_instance.customer_demands,
            customer_locations=cvrp_instance.customer_locations,
            depot_location=cvrp_instance.depot_location
        )
        
        try:
            # Solve with time limit for sensitivity analysis
            prob, result, x_vars, y_vars = solve_cvrp_mip(temp_instance, time_limit=30)
            
            if result == 1:  # Optimal solution found
                assignments, routes = extract_solution(temp_instance, x_vars, y_vars)
                metrics, total_dist = calculate_route_metrics(temp_instance, routes)
                
                # Calculate vehicles used and average utilization
                vehicles_used = sum(1 for k in range(1, temp_instance.num_vehicles + 1) 
                                 if len(assignments[k]) > 0)
                total_demand = sum(temp_instance.customer_demands)
                avg_utilization = total_demand / (vehicles_used * capacity) * 100
                
                results.append({
                    'capacity': capacity,
                    'distance': total_dist,
                    'vehicles_used': vehicles_used,
                    'avg_utilization': avg_utilization
                })
                
                print(f"{capacity:<10} {total_dist:<10.2f} {vehicles_used:<15} {avg_utilization:<10.1f}")
            else:
                print(f"{capacity:<10} {'No solution':<10} {'-':<15} {'-':<10}")
                
        except Exception as e:
            print(f"{capacity:<10} {'Error':<10} {'-':<15} {'-':<10}")
    
    return results

# Run sensitivity analysis
sensitivity_results = sensitivity_analysis_capacity(cvrp_instance)

### Why this Tier exists vs earlier Tiers
This is the foundational Tier that establishes the mathematical framework for CVRP. It provides:
- **Optimal solutions** with provable optimality guarantees
- **Mathematical understanding** of the problem structure and constraints
- **Baseline performance** for comparison with heuristic and metaheuristic methods
- **Theoretical foundation** for understanding the complexity and characteristics of CVRP

### Pros / Cons vs other approaches

**Pros:**
- Guarantees optimal solution (when solvable)
- Provides mathematical proof of optimality
- Comprehensive constraint handling
- Serves as benchmark for other methods
- Clear problem formulation and understanding

**Cons:**
- Computationally expensive for large instances
- May not scale well beyond 50-100 customers
- Requires specialized optimization software
- Sensitive to problem formulation details
- Limited practical applicability for real-time decisions

### When to use this Tier
- **Small to medium instances** (up to 50 customers)
- **Benchmark studies** to evaluate heuristic performance
- **Academic research** requiring optimal solutions
- **Strategic planning** where computational time is not critical
- **Quality-critical applications** where optimality is essential